# Haptic Ground Truth — Colab Pipeline

Same four algorithms as **[Sound2Hap](https://github.com/Iris1215/Sound2Hap)** (CHI 2026).

Convert **3–5 minute** video audio into **four candidate haptic tracks** for human-in-the-loop evaluation.

**Runtime:** CPU only — no GPU required.

**Output:** mono **8 kHz** haptic WAV files (Sound2Hap convention).

**Setup:** Run all cells top-to-bottom. No project upload needed.

**Drive / Gmail:** When you mount Drive, Colab asks **you** to sign in. Results save to **your** Google Drive — not the notebook author’s.

| Algorithm | Sound2Hap module | Method |
|-----------|------------------|--------|
| **A** | `Percept.py` | ISO loudness + roughness → dual sine (175/210 Hz) |
| **B** | `FreqShift.py` | Octave shift (-12/-24) + 10 Hz HP + 250 Hz BP |
| **C** | `Pitch_WebTool.py` | MoSQITo Bark loudness → pitch-matched sine |
| **D** | `HapticGen.py` | 10 ms RMS → 200 Hz NCO with freq offset |

In [ ]:
# Install system + Python dependencies (CPU runtime is fine)
!apt-get -qq install -y ffmpeg > /dev/null
!pip install -q numpy scipy librosa soundfile audioread resampy torch torchaudio matplotlib mosqito

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/haptic-groundtruth")
PKG_DIR = PROJECT_ROOT / "haptic_gt"
PKG_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "__init__.py": """\"\"\"Ground-truth haptic track generation from video audio.\"\"\"

from .pipeline import generate_candidate_tracks

__all__ = ["generate_candidate_tracks"]
""",
    "algorithms/__init__.py": """\"\"\"Sound2Hap signal-processing algorithms.\"\"\"
""",
    "algorithms/freq_shift.py": """\"\"\"
Frequency shifting audio-to-vibration (Sound2Hap / Okazaki et al., 2015).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

from pathlib import Path
from typing import Union

import librosa
import numpy as np
import soundfile as sf
import torch
from scipy.signal import butter, lfilter

from haptic_gt.utils.normalization import normalize_audio

VIB_SR = 8000


def _butter_bandpass(sr: int, center_hz: float = 250.0, q: float = 1.0, order: int = 4):
    bw = center_hz / q
    low_hz = max(center_hz - bw / 2, 1.0)
    high_hz = min(center_hz + bw / 2, sr / 2 - 1)
    wn = [low_hz / (sr / 2), high_hz / (sr / 2)]
    return butter(order, wn, btype="band")


def _butter_highpass(sr: int, cutoff_hz: float = 10.0, order: int = 2):
    wn = cutoff_hz / (sr / 2)
    return butter(order, wn, btype="high")


def process_file(
    in_wav: Union[str, Path],
    out_wav: Union[str, Path],
    centre_hz: float = 250.0,
    q: float = 1.0,
) -> None:
    y, sr = librosa.load(in_wav, sr=None, mono=True)

    wav_tensor = torch.from_numpy(y).float().unsqueeze(0)
    y_norm_t = normalize_audio(wav_tensor, normalize=True, strategy="peak")
    y = y_norm_t.squeeze(0).numpy()

    y_1ot = librosa.effects.pitch_shift(y, sr=sr, n_steps=-12, res_type="kaiser_best")
    y_2ot = librosa.effects.pitch_shift(y, sr=sr, n_steps=-24, res_type="kaiser_best")
    mix = y + y_1ot + y_2ot

    rms = np.sqrt(np.mean(mix**2) + 1e-12)
    mix /= rms * np.sqrt(2)

    b_hp, a_hp = _butter_highpass(sr, cutoff_hz=10.0)
    mix = lfilter(b_hp, a_hp, mix)

    b, a = _butter_bandpass(sr, centre_hz, q)
    mix_bp = lfilter(b, a, mix)
    mix_bp = librosa.resample(mix_bp, orig_sr=sr, target_sr=VIB_SR)
    mix_bp = np.clip(mix_bp, -1.0, 1.0)

    out_path = Path(out_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, mix_bp.astype(np.float32), VIB_SR, subtype="PCM_16")
""",
    "algorithms/haptic_gen.py": """\"\"\"
HapticGen-style RMS-driven NCO synthesis (Sound2Hap / Sung et al., CHI 2025).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from haptic_gt.utils.normalization import normalize_audio

WANTED_BIN_SIZE_SEC = 0.010
BASE_FREQ = 200.0
VIB_SR = 8000


def amp_env_on_wav_norm(
    wav_norm: np.ndarray,
    input_sample_rate: int,
    output_sample_rate: int,
) -> np.ndarray:
    wav_norm = wav_norm.squeeze()
    num_samples = len(wav_norm)
    duration_sec = num_samples / input_sample_rate
    samples_per_bin = int(WANTED_BIN_SIZE_SEC * input_sample_rate)
    num_bins = num_samples // samples_per_bin

    wav_chunks = np.array_split(wav_norm, num_bins)
    rms_bins = np.array([np.sqrt(np.mean(chunk**2)) for chunk in wav_chunks])
    rms_max = np.max(rms_bins)
    rms_norm = np.sqrt(2)
    rms_amplify = max(1.0, min(1.2, 1.0 / (rms_max * rms_norm)))
    rms_norm_amp = rms_norm * rms_amplify
    out_samples = int(duration_sec * output_sample_rate)

    phase_acc = 0.0
    output = np.zeros(out_samples)
    for i in range(out_samples):
        t = i / output_sample_rate
        t_prog = t / duration_sec
        bin_fi = t_prog * num_bins
        bin_lo = int(bin_fi)
        bin_hi = min(num_bins - 1, int(math.ceil(bin_fi)))
        bin_fr = bin_fi - bin_lo
        rms_val = (
            rms_bins[bin_lo] * (1.0 - bin_fr) + rms_bins[bin_hi] * bin_fr
        ) * rms_norm_amp
        freq_offset = (rms_val - 0.3) * 100.0
        phase_delta = 2.0 * math.pi * (BASE_FREQ + freq_offset) / output_sample_rate
        phase_acc = (phase_acc + phase_delta) % (2.0 * math.pi)
        output[i] = rms_val * math.sin(phase_acc)

    return output


def process_file(input_path: str | Path, output_path: str | Path) -> None:
    wav_data, sr = sf.read(input_path)
    wav_tensor = torch.from_numpy(wav_data).float().unsqueeze(0)
    wav_norm_tensor = normalize_audio(
        wav_tensor,
        normalize=True,
        strategy="peak",
        peak_clip_headroom_db=0,
        peak_normalize_db_clamp=0,
    )
    wav = wav_norm_tensor.squeeze(0).numpy()
    env_signal = amp_env_on_wav_norm(wav, sr, VIB_SR)

    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, env_signal, VIB_SR, subtype="PCM_16")
""",
    "algorithms/percept.py": """\"\"\"
Perception-level audio-to-vibration translator (Sound2Hap / Lee & Choi, CHI 2013).

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from scipy.signal import find_peaks

from haptic_gt.utils.normalization import normalize_audio

AUDIO_SR = 44100
VIB_SR = 8000
FRAME_S = 4096
F1 = 175.0
F2 = 210.0
CR = 0.035
OR = 0.40
CV = 1
CL = 0.1
OL = 3.8
C_FULLBAND = 0.065
F_FULLBAND = 6400
C_BASS = 1.91
F_BASS = 200
C = 1.37

_ISO_FREQ = np.array(
    [
        25, 31.5, 40, 50, 63, 80, 100, 125, 160, 200, 250, 315, 400, 500, 630,
        800, 1000, 1250, 1600, 2000, 2500, 3150, 4000, 5000, 6300,
    ]
)
_ISO_SPL60 = np.array(
    [
        104.23, 99.08, 94.18, 89.96, 85.94, 82.05, 78.65, 75.56, 72.47, 69.86,
        67.53, 65.39, 63.45, 62.05, 60.81, 59.89, 60.01, 62.15, 63.19, 59.96,
        57.26, 56.42, 57.57, 60.89, 66.36,
    ]
)


def iso60phon(f: np.ndarray) -> np.ndarray:
    return np.interp(f, _ISO_FREQ, _ISO_SPL60, left=_ISO_SPL60[0], right=_ISO_SPL60[-1])


def auditory_loudness(frame: np.ndarray, content: str) -> float:
    if content == "music":
        c_use, f_max = C_BASS, F_BASS
    else:
        c_use, f_max = C_FULLBAND, F_FULLBAND

    mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(frame.size, 1 / AUDIO_SR)
    mask = (freqs >= 25) & (freqs <= f_max)
    mag = mag[mask]
    freqs = freqs[mask]

    db = 20 * np.log10(C * mag + 1e-12)
    af = iso60phon(freqs)
    loudness = c_use * np.sum(db / af)
    return max(0.0, loudness)


def auditory_roughness(frame: np.ndarray, peak_db: float = -40.0) -> float:
    mag = np.abs(np.fft.rfft(frame))
    freqs = np.fft.rfftfreq(frame.size, 1 / AUDIO_SR)
    mask = (freqs >= 25) & (freqs <= 6400)
    mag = mag[mask]
    freqs = freqs[mask]

    db = 20 * np.log10(mag + 1e-12)
    thresh = db.max() + peak_db
    peaks, _ = find_peaks(db, height=thresh)

    f = freqs[peaks]
    x = mag[peaks]
    roughness = 0.0
    for i in range(len(f)):
        for j in range(i + 1, len(f)):
            f1, f2 = f[i], f[j]
            x1, x2 = x[i], x[j]
            xm, xM = min(x1, x2), max(x1, x2)
            fd = abs(f2 - f1)
            s = 0.24 / (0.0207 * min(f1, f2) + 18.96)
            term = ((xm * xM) ** 0.1 / 2.0) * (2 * xm / (xm + xM)) ** 3.11
            roughness += term * (math.exp(-3.5 * s * fd) - math.exp(-5.75 * s * fd))
    return roughness


def perceptual_targets(la: float, ra: float, content: str) -> tuple[float, float]:
    if content == "music":
        iv = CL * la - OL
    else:
        iv = CR * math.sqrt(la) * (ra**2) - OR
    rv = CV * ra
    return max(0, iv), rv


def amplitudes_from_percepts(iv: float, rv: float) -> tuple[float, float]:
    if iv <= 0.0:
        return 0.0, 0.0

    rv_max = (801.0 / 113.0) + 0.529 * iv + 0.479
    rv_adj = min(rv, rv_max)
    disc = max(0.0, 801.0 - 113.0 * (rv_adj - 0.529 * iv - 0.479))
    r1 = (28.3 + math.sqrt(disc)) / 56.3
    r2 = (28.3 - math.sqrt(disc)) / 56.3
    valid = [s for s in (r1, r2) if 0.0 <= s <= 1.0]
    s = min(valid) if valid else (28.3 / 56.3)

    a = ((25.8 * s**2 - 25.5 * s + rv_adj - 0.203) / 3.98) ** 2
    a2 = a * s
    a1 = a - a2
    return a1, a2


def synth_vibration(a1: float, a2: float, n_samples: int) -> np.ndarray:
    t = np.arange(n_samples) / VIB_SR
    return a1 * np.sin(2 * math.pi * F1 * t) + a2 * np.sin(2 * math.pi * F2 * t)


def read_wav_mono_44k(fname: str | Path) -> np.ndarray:
    wav_data, _ = sf.read(fname)
    if wav_data.ndim > 1:
        wav_data = wav_data.mean(axis=1)
    wav_tensor = torch.from_numpy(wav_data).float().unsqueeze(0)
    wav_norm_tensor = normalize_audio(
        wav_tensor,
        normalize=True,
        strategy="peak",
        peak_clip_headroom_db=0,
        peak_normalize_db_clamp=0,
    )
    return wav_norm_tensor.squeeze(0).numpy().astype("float32")


def process_file(
    in_wav: str | Path,
    out_wav: str | Path,
    content: str = "game",
) -> None:
    audio = read_wav_mono_44k(in_wav)
    hop_s = FRAME_S
    n_out_total = int(np.ceil(len(audio) * VIB_SR / AUDIO_SR))
    vib_full = np.zeros(n_out_total, dtype=np.float32)

    for start in range(0, len(audio), hop_s):
        block = audio[start : start + FRAME_S]
        if block.size == 0:
            break
        if block.size < FRAME_S:
            block = np.pad(block, (0, FRAME_S - block.size), "constant")

        la = auditory_loudness(block, content)
        ra = auditory_roughness(block)
        iv, rv = perceptual_targets(la, ra, content)
        a1, a2 = amplitudes_from_percepts(iv, rv)

        n_out = int(round(FRAME_S * VIB_SR / AUDIO_SR))
        vib_seg = synth_vibration(a1, a2, n_out)
        rms_seg = np.sqrt(np.mean(vib_seg**2) + 1e-12)
        vib_seg /= rms_seg * np.sqrt(2)

        out_start = int(round(start * VIB_SR / AUDIO_SR))
        out_end = out_start + n_out
        if out_end > n_out_total:
            vib_full[out_start:] += vib_seg[: n_out_total - out_start]
        else:
            vib_full[out_start:out_end] += vib_seg

    vib_full = np.clip(vib_full, -1.0, 1.0)
    out_path = Path(out_wav)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out_path, vib_full, VIB_SR, subtype="PCM_16")
""",
    "algorithms/pitch_match.py": """\"\"\"
Pitch Match audio-to-vibration (Sound2Hap / Kim et al., IEEE ToH 2023).

Python version used for Sound2Hap web tool. MATLAB version used in the study.

Adapted from: https://github.com/Iris1215/Sound2Hap
\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import get_window, resample_poly

try:
    from mosqito.functions.loudness_zwtv._loudness_zwtv import loudness_zwtv

    MOSQITO_AVAILABLE = True
except Exception:
    MOSQITO_AVAILABLE = False

VIB_SR = 8000


@dataclass
class Config:
    regressionCoeffs: dict
    vibrationFreqRange: tuple
    binSizeMs: float
    overlapRatio: float
    smoothingWindow: int
    inputSampleRate: int
    outputSampleRate: int


def get_config() -> Config:
    return Config(
        regressionCoeffs={2: -0.005, 3: 0.003, 9: -0.015, 12: 0.008, 24: 0.008},
        vibrationFreqRange=(50.0, 398.0),
        binSizeMs=10.0,
        overlapRatio=0.5,
        smoothingWindow=3,
        inputSampleRate=44100,
        outputSampleRate=VIB_SR,
    )


def normalize_audio(audio: np.ndarray, do_normalize: bool) -> np.ndarray:
    scale_peak = 10 ** (-1 / 20)
    normalize_peak = 1.0
    wav_max = np.max(np.abs(audio)) + 1e-12
    rescaling = min(max(1.0, normalize_peak / wav_max), scale_peak / wav_max)
    if do_normalize or (rescaling < 1.0):
        audio = audio * rescaling
    return audio


def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x)) + 1e-12))


def _specific_and_total_loudness_bark(audio_bin: np.ndarray, sr: int):
    if not MOSQITO_AVAILABLE:
        env = np.abs(audio_bin)
        total = float(np.mean(env))
        spec24 = np.zeros(24, dtype=np.float32)
        spec24[0] = total
        return spec24, total

    try:
        results = loudness_zwtv(audio_bin, sr, field_type="free")
        n_time = np.asarray(results["N"]).reshape(-1)
        n_spec = np.asarray(results["N_specific"])
        total_loudness = float(np.mean(n_time)) if n_time.size else 0.0

        if n_spec.ndim == 2 and n_spec.shape[1] >= 240:
            spec_time_mean = np.mean(n_spec, axis=0)
            spec24 = np.zeros(24, dtype=np.float32)
            for i in range(24):
                start = i * 10
                end = start + 10
                spec24[i] = float(np.sum(spec_time_mean[start:end]))
        else:
            if n_spec.ndim == 1:
                vec = n_spec
            else:
                vec = np.mean(n_spec, axis=0) if n_spec.size else np.zeros(240)
            idx = np.linspace(0, len(vec) - 1, 24)
            spec24 = np.interp(idx, np.arange(len(vec)), vec).astype(np.float32)

        spec24[~np.isfinite(spec24)] = 0.0
        return spec24, total_loudness
    except Exception:
        env = np.abs(audio_bin)
        total = float(np.mean(env))
        spec24 = np.zeros(24, dtype=np.float32)
        spec24[0] = total
        return spec24, total


def predict_vibration_frequency(specific_loudness_24: np.ndarray, cfg: Config) -> float:
    predicted = 0.0
    for bark_band, coeff in cfg.regressionCoeffs.items():
        idx = int(bark_band) - 1
        if 0 <= idx < len(specific_loudness_24):
            predicted += coeff * float(specific_loudness_24[idx]) * 1000.0
    predicted = abs(predicted)
    vmin, vmax = cfg.vibrationFreqRange
    return float(np.clip(predicted, vmin, vmax))


def analyze_audio_bins(audio: np.ndarray, sr: int, cfg: Config):
    bin_size = int(round(cfg.binSizeMs * sr / 1000.0))
    hop = max(1, int(round(bin_size * (1.0 - cfg.overlapRatio))))
    if bin_size < 2:
        bin_size = 2
    starts = np.arange(0, max(1, len(audio) - bin_size + 1), hop, dtype=int)
    if starts.size == 0:
        starts = np.array([0], dtype=int)

    times = (starts + bin_size / 2.0) / float(sr)
    freqs = np.zeros(starts.size, dtype=np.float32)
    amps = np.zeros(starts.size, dtype=np.float32)
    win = get_window("hann", bin_size, fftbins=False).astype(np.float32)

    for i, s in enumerate(starts):
        e = min(s + bin_size, len(audio))
        chunk = np.zeros(bin_size, dtype=np.float32)
        seg = audio[s:e]
        chunk[: len(seg)] = seg
        chunk *= win

        if rms(chunk) < 1e-3:
            freqs[i] = freqs[i - 1] if i > 0 else np.mean(cfg.vibrationFreqRange)
            amps[i] = 0.0
            continue

        spec24, loud = _specific_and_total_loudness_bark(chunk, sr)
        freqs[i] = predict_vibration_frequency(spec24, cfg)
        amps[i] = float(loud)

    if cfg.smoothingWindow > 1 and len(freqs) > cfg.smoothingWindow:
        k = cfg.smoothingWindow
        kernel = np.ones(k, dtype=np.float32) / k
        freqs = np.convolve(freqs, kernel, mode="same")

    return times.astype(np.float64), freqs.astype(np.float64), amps.astype(np.float64)


def generate_time_varying_vibration(audio: np.ndarray, sr: int, cfg: Config):
    bin_t, bin_f, bin_a = analyze_audio_bins(audio, sr, cfg)
    t = np.arange(len(audio), dtype=np.float64) / float(sr)

    if len(bin_t) == 1:
        f_inst = np.full_like(t, bin_f[0], dtype=np.float64)
        a_inst = np.full_like(t, bin_a[0], dtype=np.float64)
    else:
        try:
            from scipy.interpolate import PchipInterpolator

            f_inst = PchipInterpolator(bin_t, bin_f, extrapolate=True)(t)
        except Exception:
            f_inst = np.interp(t, bin_t, bin_f, left=bin_f[0], right=bin_f[-1])
        a_inst = np.interp(t, bin_t, bin_a, left=bin_a[0], right=bin_a[-1])

    rms_current = rms(a_inst)
    rms_norm = np.sqrt(2.0)
    rms_amplify = max(1.0, min(1.2, 1.0 / (rms_current * rms_norm))) if rms_current > 0 else 1.0
    a_inst = a_inst * (rms_norm * rms_amplify) if rms_current > 0 else np.full_like(t, 0.1)

    dt = 1.0 / float(sr)
    phi = np.empty_like(t)
    phi[0] = 0.0
    phi[1:] = 2.0 * np.pi * np.cumsum(f_inst[:-1]) * dt
    v = a_inst * np.sin(phi)

    fade_len = int(round(0.01 * sr))
    if len(v) > 2 * fade_len and fade_len > 0:
        fade_in = np.linspace(0.0, 1.0, fade_len)
        fade_out = np.linspace(1.0, 0.0, fade_len)
        v[:fade_len] *= fade_in
        v[-fade_len:] *= fade_out

    return v.astype(np.float32), f_inst.astype(np.float32), a_inst.astype(np.float32)


def generate_vibration_signal(audio: np.ndarray, sr: int, cfg: Config):
    v, f_arr, a_arr = generate_time_varying_vibration(audio, sr, cfg)
    analysis_info = {
        "method": "time_varying",
        "duration": len(audio) / float(sr),
        "freqMean": float(np.mean(f_arr)),
        "freqRange": (float(np.min(f_arr)), float(np.max(f_arr))),
        "freqStd": float(np.std(f_arr)),
    }
    return v, f_arr, a_arr, analysis_info


def _read_mono(path: str | Path):
    x, sr = sf.read(path, always_2d=False)
    x = x.astype(np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    return x, sr


def _write_int16_wav(path: str | Path, y: np.ndarray, sr: int):
    y = y / (np.max(np.abs(y)) + 1e-12)
    sf.write(path, y, sr, subtype="PCM_16")


def process_file(
    input_file: str | Path,
    output_file: str | Path,
    cfg: Config | None = None,
) -> dict:
    cfg = cfg or get_config()
    audio, sr = _read_mono(input_file)
    duration = len(audio) / float(sr)
    audio = normalize_audio(audio, True)

    v, f_arr, a_arr, info = generate_vibration_signal(audio, sr, cfg)
    fs_out = cfg.outputSampleRate
    if sr != fs_out:
        frac = Fraction(fs_out, sr).limit_denominator(1000)
        v = resample_poly(v, frac.numerator, frac.denominator)

    out_path = Path(output_file)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    _write_int16_wav(out_path, v, fs_out)

    return {
        "inputFile": str(input_file),
        "outputFile": str(output_file),
        "duration": duration,
        "originalSr": sr,
        "targetSr": fs_out,
        "analysisInfo": info,
        "mosqitoAvailable": MOSQITO_AVAILABLE,
    }
""",
    "audio_io.py": """\"\"\"Audio extraction, loading, and export (Sound2Hap-compatible rates).\"\"\"

from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

INPUT_SR = 44_100
VIB_SR = 8_000


def _require_ffmpeg() -> str:
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError(
            "ffmpeg not found. Install it first "
            "(Colab: !apt-get -qq install ffmpeg)."
        )
    return ffmpeg


def extract_audio_from_video(
    video_path: str | Path,
    output_path: str | Path,
    sr: int = INPUT_SR,
) -> Path:
    \"\"\"Extract mono 16-bit PCM WAV at 44.1 kHz from a video file.\"\"\"
    video_path = Path(video_path)
    output_path = Path(output_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    ffmpeg = _require_ffmpeg()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        ffmpeg,
        "-y",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(sr),
        "-sample_fmt",
        "s16",
        str(output_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\\n{result.stderr}")
    return output_path


def prepare_source_wav(
    audio_path: str | Path,
    output_path: str | Path,
    sr: int = INPUT_SR,
) -> Path:
    \"\"\"Convert/load audio to mono 16-bit PCM WAV at 44.1 kHz.\"\"\"
    audio_path = Path(audio_path)
    output_path = Path(output_path)
    if not audio_path.exists():
        raise FileNotFoundError(f"Audio not found: {audio_path}")

    audio, _ = librosa.load(audio_path, sr=sr, mono=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(output_path, np.clip(audio, -1.0, 1.0), sr, subtype="PCM_16")
    return output_path


def save_haptic(path: str | Path, audio: np.ndarray, sr: int = VIB_SR) -> Path:
    \"\"\"Write a mono haptic track as 16-bit PCM WAV (default 8 kHz).\"\"\"
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, np.clip(audio, -1.0, 1.0), sr, subtype="PCM_16")
    return path
""",
    "pipeline.py": """\"\"\"End-to-end pipeline aligned with Sound2Hap signal processing.\"\"\"

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

from haptic_gt.algorithms import freq_shift, haptic_gen, percept, pitch_match
from haptic_gt.audio_io import INPUT_SR, VIB_SR, extract_audio_from_video, prepare_source_wav

OUTPUT_NAMES = {
    "source_audio": "source_audio.wav",
    "algorithm_a_perception_mapping": "algorithm_a_perception_mapping.wav",
    "algorithm_b_frequency_shifting": "algorithm_b_frequency_shifting.wav",
    "algorithm_c_pitch_matching": "algorithm_c_pitch_matching.wav",
    "algorithm_d_haptic_gen": "algorithm_d_haptic_gen.wav",
}


@dataclass
class CandidateTracks:
    \"\"\"Paths to source audio and four Sound2Hap candidate haptic tracks.\"\"\"

    source_wav: Path
    algorithm_a: Path
    algorithm_b: Path
    algorithm_c: Path
    algorithm_d: Path
    output_dir: Path
    input_sample_rate: int = INPUT_SR
    output_sample_rate: int = VIB_SR
    pitch_match_info: dict | None = None

    def save_all(self) -> dict[str, Path]:
        return {
            "source_audio": self.source_wav,
            "algorithm_a_perception_mapping": self.algorithm_a,
            "algorithm_b_frequency_shifting": self.algorithm_b,
            "algorithm_c_pitch_matching": self.algorithm_c,
            "algorithm_d_haptic_gen": self.algorithm_d,
        }


def generate_candidate_tracks(
    input_path: str | Path,
    output_dir: str | Path,
    *,
    from_video: bool = True,
    content_type: str = "game",
) -> CandidateTracks:
    \"\"\"
    Run the Sound2Hap processing engine on one video or audio file.

    Parameters
    ----------
    input_path:
        Video (mp4, mov, ...) or audio (wav, mp3, ...) path.
    output_dir:
        Directory for 44.1 kHz source + 8 kHz haptic WAV outputs.
    from_video:
        Extract audio with ffmpeg when True.
    content_type:
        Perceptual mapping content profile: ``"game"`` (games/movies) or ``"music"``.
    \"\"\"
    input_path = Path(input_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    source_wav = output_dir / OUTPUT_NAMES["source_audio"]
    if from_video:
        extract_audio_from_video(input_path, source_wav, sr=INPUT_SR)
    else:
        prepare_source_wav(input_path, source_wav, sr=INPUT_SR)

    out_a = output_dir / OUTPUT_NAMES["algorithm_a_perception_mapping"]
    out_b = output_dir / OUTPUT_NAMES["algorithm_b_frequency_shifting"]
    out_c = output_dir / OUTPUT_NAMES["algorithm_c_pitch_matching"]
    out_d = output_dir / OUTPUT_NAMES["algorithm_d_haptic_gen"]

    percept.process_file(source_wav, out_a, content=content_type)
    freq_shift.process_file(source_wav, out_b)
    pitch_info = pitch_match.process_file(source_wav, out_c)
    haptic_gen.process_file(source_wav, out_d)

    return CandidateTracks(
        source_wav=source_wav,
        algorithm_a=out_a,
        algorithm_b=out_b,
        algorithm_c=out_c,
        algorithm_d=out_d,
        output_dir=output_dir,
        pitch_match_info=pitch_info,
    )
""",
    "utils/__init__.py": """\"\"\"Utils package.\"\"\"
""",
    "utils/normalization.py": """\"\"\"Peak/RMS/loudness normalization (from Sound2Hap).\"\"\"

from __future__ import annotations

import sys
import typing as tp

import torch
import torchaudio


def normalize_loudness(
    wav: torch.Tensor,
    sample_rate: int,
    loudness_headroom_db: float = 14,
    loudness_compressor: bool = False,
    energy_floor: float = 2e-3,
) -> torch.Tensor:
    energy = wav.pow(2).mean().sqrt().item()
    if energy < energy_floor:
        return wav
    transform = torchaudio.transforms.Loudness(sample_rate)
    input_loudness_db = transform(wav).item()
    delta_loudness = -loudness_headroom_db - input_loudness_db
    gain = 10.0 ** (delta_loudness / 20.0)
    output = gain * wav
    if loudness_compressor:
        output = torch.tanh(output)
    return output


def _clip_wav(
    wav: torch.Tensor,
    log_clipping: bool = False,
    stem_name: tp.Optional[str] = None,
) -> None:
    max_scale = wav.abs().max()
    if log_clipping and max_scale > 1:
        clamp_prob = (wav.abs() > 1).float().mean().item()
        print(
            f"CLIPPING {stem_name or ''} happening with proba:",
            clamp_prob,
            "maximum scale:",
            max_scale.item(),
            file=sys.stderr,
        )
    wav.clamp_(-1, 1)


def normalize_audio(
    wav: torch.Tensor,
    normalize: bool = True,
    strategy: str = "peak",
    peak_clip_headroom_db: float = 1,
    peak_normalize_db_clamp: float = 0,
    rms_headroom_db: float = 18,
    loudness_headroom_db: float = 14,
    loudness_compressor: bool = False,
    log_clipping: bool = False,
    sample_rate: tp.Optional[int] = None,
    stem_name: tp.Optional[str] = None,
) -> torch.Tensor:
    scale_peak = 10 ** (-peak_clip_headroom_db / 20)
    normalize_peak = 10 ** (peak_normalize_db_clamp / 20)
    scale_rms = 10 ** (-rms_headroom_db / 20)
    if strategy == "peak":
        wav_max = wav.abs().max()
        rescaling = (scale_peak / wav_max).clamp(max=(normalize_peak / wav_max).clamp(min=1))
        if normalize or rescaling < 1:
            wav = wav * rescaling
    elif strategy == "clip":
        wav = wav.clamp(-scale_peak, scale_peak)
    elif strategy == "rms":
        mono = wav.mean(dim=0)
        rescaling = scale_rms / mono.pow(2).mean().sqrt()
        if normalize or rescaling < 1:
            wav = wav * rescaling
        _clip_wav(wav, log_clipping=log_clipping, stem_name=stem_name)
    elif strategy == "loudness":
        assert sample_rate is not None, "Loudness normalization requires sample rate."
        wav = normalize_loudness(
            wav, sample_rate, loudness_headroom_db, loudness_compressor
        )
        _clip_wav(wav, log_clipping=log_clipping, stem_name=stem_name)
    else:
        assert wav.abs().max() < 1
        assert strategy in ("", "none"), f"Unexpected strategy: '{strategy}'"
    return wav
"""

}

for name, source in FILES.items():
    target = PKG_DIR / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

sys.path.insert(0, str(PROJECT_ROOT))
print("Installed haptic_gt at", PKG_DIR)
print("Modules:", ", ".join(sorted(FILES)))


In [ ]:
# Mount THIS runner's Google Drive (whoever is signed into Colab right now)
from google.colab import auth, drive
import subprocess
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

# Optional: show which Google account is connected
try:
    auth.authenticate_user()
    RUNNER_EMAIL = subprocess.check_output(
        ["gcloud", "config", "get-value", "account"], text=True
    ).strip()
except Exception:
    RUNNER_EMAIL = "(could not read account — Drive is still mounted for the current Colab user)"

# Folder on THE RUNNER's Drive (not a hardcoded personal path)
DRIVE_ROOT = Path("/content/drive/MyDrive/haptic-groundtruth")
INPUT_DIR = DRIVE_ROOT / "inputs"
OUTPUT_DIR = DRIVE_ROOT / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Signed-in account:", RUNNER_EMAIL)
print("Your Drive folder:", DRIVE_ROOT)
print("Put videos in:", INPUT_DIR)
print("Results go to:", OUTPUT_DIR)

In [ ]:
from google.colab import files
from IPython.display import Audio, display
from haptic_gt.pipeline import generate_candidate_tracks

# Prefer a video already on THIS runner's Drive; otherwise upload into their Drive folder
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}
drive_videos = sorted(
    p for p in INPUT_DIR.iterdir() if p.suffix.lower() in VIDEO_EXTS
) if INPUT_DIR.exists() else []

if drive_videos:
    print(f"Found {len(drive_videos)} video(s) in your Drive folder:")
    for i, p in enumerate(drive_videos):
        print(f"  [{i}] {p.name}")
    choice = input("Enter index to use, or press Enter to upload a new file: ").strip()
    if choice != "":
        video_path = drive_videos[int(choice)]
    else:
        print("Upload a video (saved to YOUR Drive inputs folder)...")
        uploaded = files.upload()
        video_name = next(iter(uploaded))
        src = Path("/content") / video_name
        video_path = INPUT_DIR / video_name
        video_path.write_bytes(src.read_bytes())
else:
    print("No videos in your Drive yet. Upload one (saved to YOUR Drive)...")
    uploaded = files.upload()
    video_name = next(iter(uploaded))
    src = Path("/content") / video_name
    video_path = INPUT_DIR / video_name
    video_path.write_bytes(src.read_bytes())

print("Using video:", video_path)
print("Account:", RUNNER_EMAIL)
print("Output folder (your Drive):", OUTPUT_DIR)


In [ ]:
%%time
import soundfile as sf

CONTENT_TYPE = "game"  # "game" for games/movies, "music" for music

tracks = generate_candidate_tracks(
    video_path,
    OUTPUT_DIR,
    from_video=True,
    content_type=CONTENT_TYPE,
)
saved = tracks.save_all()

src_audio, src_sr = sf.read(saved["source_audio"])
print(f"Source: {len(src_audio)/src_sr:.1f}s @ {src_sr} Hz")
print(f"Haptic outputs @ {tracks.output_sample_rate} Hz")
if tracks.pitch_match_info:
    print(f"Pitch match MoSQITo: {tracks.pitch_match_info.get('mosqitoAvailable')}")
print("Saved files:")
for name, path in saved.items():
    print(f"  {name}: {path}")

In [ ]:
labels = {
    'source_audio': 'Source audio',
    'algorithm_a_perception_mapping': 'A — Perception mapping',
    'algorithm_b_frequency_shifting': 'B — Frequency shifting',
    'algorithm_c_pitch_matching': 'C — Pitch matching',
    'algorithm_d_haptic_gen': 'D — HapticGen',
}

for key, label in labels.items():
    print(label)
    display(Audio(str(saved[key])))

In [ ]:
import shutil

# Save ZIP on THIS runner's Drive, and also offer browser download
zip_base = DRIVE_ROOT / "haptic_candidates"
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR))
print("Saved to your Drive:", zip_path)
print("Account:", RUNNER_EMAIL)

# Optional local browser download
files.download(str(zip_path))
print("Download started.")


## Human-in-the-loop (next step)

1. Play **source audio** (44.1 kHz) in headphones while feeling each **8 kHz** candidate on haptic hardware.
2. Rate **realism** and **similarity** (e.g. 1–7 Likert) per algorithm — same protocol as [Sound2Hap](https://sound2hap.netlify.app/).
3. If agreement < threshold, tune parameters in `haptic_gt/algorithms/*.py` and re-run.
4. Approved tracks become your **ground truth dataset**.